# Detección de Fraude — Entrenamiento Binario (LightGBM + XGBoost)

**Objetivo:** Maximizar recall de la clase fraude manteniendo precisión ≥ 60%.
**Dataset sintético (Faker):** ~10,000 transacciones, ~15% fraude.
**Estrategia:** 2 modelos base fuertes, threshold optimizado en validación, evaluación final en test.

## 1. Carga y exploración

In [ ]:

import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score,
    precision_score, recall_score, make_scorer, fbeta_score)
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
import lightgbm as lgb
sys.path.append(str(Path.cwd().parent))
from model.feature_engineering import FeatureEngineer
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)
print('OK')


In [ ]:

DATA_PATH = Path.cwd().parent / 'Notebooks' / 'data' / 'dataset_fraude.csv'
df = pd.read_csv(DATA_PATH)
print(f'Filas: {df.shape[0]:,}  Columnas: {df.shape[1]}')

# Prevent data leakage: drop the other target column immediately
other_target = [c for c in ['IS_FRAUD', 'IMPACTO_FRAUDE'] if c != 'IS_FRAUD']
if other_target:
    df = df.drop(columns=other_target, errors='ignore')
    print(f'Dropped leakage column: {other_target[0]}')

print(f'Fraude: {df["IS_FRAUD"].sum():,} / {len(df):,} ({df["IS_FRAUD"].mean()*100:.2f}%)')
print(f'Balance: {df["IS_FRAUD"].value_counts().to_dict()}')


## 2. Feature Engineering

Features derivadas de la `FeatureEngineer`:
- **Raw:** importe_transaccion, saldo_actual, hora, día_semana, país, etc.
- **Derived:** ratios (txn vs límite, txn vs saldo, outflow/inflow), log-transforms, cross-border flags, frecuencia de categorías, target encoding.

**Anti-leakage:** No se usan variables que filtren la etiqueta. `IMPACTO_FRAUDE` ya se eliminó.

In [ ]:

fe = FeatureEngineer(encode_target='IS_FRAUD', random_state=42)
X_fe = fe.fit_transform(df)
y = X_fe.pop('IS_FRAUD').values

print(f'Features totales: {X_fe.shape[1]}')
print(f'Features numéricas: {len(X_fe.select_dtypes(include=[np.number]).columns)}')


## 3. Train / Validation / Test Split

División estratificada 60/20/20. El threshold se elige **exclusivamente en validación**. Test solo para reporte final.

In [ ]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X_fe, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train:  {X_train.shape[0]:,}  ({y_train.mean()*100:.2f}% fraude)')
print(f'Val:    {X_val.shape[0]:,}  ({y_val.mean()*100:.2f}% fraude)')
print(f'Test:   {X_test.shape[0]:,}  ({y_test.mean()*100:.2f}% fraude)')


In [ ]:

# Scale numerical features (fit only on train)
num_feats = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train_s = X_train.copy()
X_train_s[num_feats] = scaler.fit_transform(X_train[num_feats])
X_val_s = X_val.copy()
X_val_s[num_feats] = scaler.transform(X_val[num_feats])
X_test_s = X_test.copy()
X_test_s[num_feats] = scaler.transform(X_test[num_feats])
print(f'Escalado completado: {len(num_feats)} features numéricas')


## 4. LightGBM — Entrenamiento y Tuning

LightGBM es ideal para fraude: maneja desbalanceo nativamente con `class_weight`, es rápido, y tiene buena regularización.

**Grid de hiperparámetros clave para fraude:**
- `scale_pos_weight`: ~ratio no-fraude/fraude ≈ 5.5
- `num_leaves`, `min_child_samples`: controlan complejidad
- `subsample`, `colsample_bytree`: regularización
- `reg_lambda`, `reg_alpha`: regularización L2/L1
- `learning_rate`, `n_estimators`: compromiso velocidad/precisión

**Scoring:** `average_precision` (PR-AUC) porque optimiza directamente el ranking de la clase minoritaria.

In [ ]:
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Scale pos weight sugerido: {scale:.2f}')

# Grid reducido para tiempos razonables (~300 combos × 5-fold)
lgb_params = {
    'scale_pos_weight': [scale * 0.8, scale, scale * 1.2],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [20, 100],
    'subsample': [0.8, 1.0],
    'reg_lambda': [0, 10],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [200, 400],
}

lgb_base = lgb.LGBMClassifier(random_state=42, verbose=-1)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

lgb_grid = GridSearchCV(
    lgb_base, lgb_params, cv=cv, scoring='average_precision',
    n_jobs=-1, verbose=1
)
lgb_grid.fit(X_train_s, y_train)
lgb_best = lgb_grid.best_estimator_

print(f'LightGBM best CV PR-AUC: {lgb_grid.best_score_:.4f}')
print(f'Best params: {lgb_grid.best_params_}')


## 5. XGBoost — Entrenamiento y Tuning

XGBoost complementa a LightGBM: usa boosting por nivel (level-wise) vs hoja (leaf-wise), maneja desbalanceo con `scale_pos_weight`, y tiene diferente comportamiento en regularización.

In [ ]:
xgb_params = {
    'scale_pos_weight': [scale * 0.8, scale, scale * 1.2],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 5],
    'subsample': [0.8, 1.0],
    'reg_lambda': [0, 10],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [200, 400],
}

xgb_base = xgb.XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss')
xgb_grid = GridSearchCV(
    xgb_base, xgb_params, cv=cv, scoring='average_precision',
    n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train_s, y_train)
xgb_best = xgb_grid.best_estimator_

print(f'XGBoost best CV PR-AUC: {xgb_grid.best_score_:.4f}')
print(f'Best params: {xgb_grid.best_params_}')


## 6. Comparación en Validación

Evaluamos ambos modelos en validación con PR-AUC, AUC-ROC, y Precision@Recall=0.9.

In [ ]:

def evaluate_model(name, model, X, y):
    yprob = model.predict_proba(X)[:, 1]
    pauc = average_precision_score(y, yprob)
    auc = roc_auc_score(y, yprob)
    prec, rec, thrs = precision_recall_curve(y, yprob)
    # Precision@Recall=0.9: max precision when recall >= 0.9
    pr90 = 0.0
    for p, r in zip(prec[:-1], rec[:-1]):
        if r >= 0.90 and p > pr90:
            pr90 = p
    return {'model': name, 'PR-AUC': pauc, 'AUC-ROC': auc, 'Prec@Rec90': pr90}

results_val = []
for name, model in [('LightGBM', lgb_best), ('XGBoost', xgb_best)]:
    r = evaluate_model(name, model, X_val_s, y_val)
    results_val.append(r)
    print(f'{name}:  PR-AUC={r["PR-AUC"]:.4f}  AUC-ROC={r["AUC-ROC"]:.4f}  Prec@Rec90={r["Prec@Rec90"]:.4f}')


## 7. Calibración de Probabilidades

Evaluamos si la calibración mejora la estabilidad del threshold. La calibración (Platt scaling) se hace con 3-fold CV sobre train, sin fuga de datos.

In [ ]:

def calibrate_model(name, model, X, y):
    cal = CalibratedClassifierCV(model, cv=3, method='sigmoid')
    cal.fit(X, y)
    return cal

cal_lgb = calibrate_model('LightGBM', lgb_best, X_train_s, y_train)
cal_xgb = calibrate_model('XGBoost', xgb_best, X_train_s, y_train)

# Compare calibrated vs raw on validation
for raw_model, cal_model, name in [(lgb_best, cal_lgb, 'LightGBM'), (xgb_best, cal_xgb, 'XGBoost')]:
    yprob_raw = raw_model.predict_proba(X_val_s)[:, 1]
    yprob_cal = cal_model.predict_proba(X_val_s)[:, 1]
    pauc_raw = average_precision_score(y_val, yprob_raw)
    pauc_cal = average_precision_score(y_val, yprob_cal)
    print(f'{name}:  Raw PR-AUC={pauc_raw:.4f}  Calibrated PR-AUC={pauc_cal:.4f}')


**Decisión sobre calibración:** Si la PR-AUC cambia poco (diferencia < 0.01), usamos probabilidades raw para threshold, ya que:
1. La calibración no altera el ranking de probabilidades significativamente.
2. El threshold se elige sobre validation, no sobre el valor 0.5.
3. Simplifica el pipeline de producción al evitar un paso extra.

Si la calibración mejora, se usan probabilidades calibradas.

## 8. Selección de Threshold en Validación

Regla: **maximizar recall sujeto a precisión ≥ 0.60**.

Si ningún threshold alcanza precisión ≥ 0.60, se reporta el mejor compromiso posible.

In [ ]:
# Pick best model
best_name = 'LightGBM' if results_val[0]['PR-AUC'] >= results_val[1]['PR-AUC'] else 'XGBoost'
best_model = lgb_best if best_name == 'LightGBM' else xgb_best
print(f'Modelo seleccionado: {best_name}')

# Compute validation probabilities
yprob_val = best_model.predict_proba(X_val_s)[:, 1]

# Threshold search on validation
target_precision = 0.60
thrs = np.linspace(0.01, 0.99, 500)
best_t_60 = None; best_rec_60 = -1; best_prec_60 = 0.0

results_table = []
for t in thrs:
    yt = (yprob_val >= t).astype(int)
    p = precision_score(y_val, yt, zero_division=0)
    r = recall_score(y_val, yt, zero_division=0)
    results_table.append({'threshold': t, 'precision': p, 'recall': r})
    if p >= target_precision and r > best_rec_60:
        best_t_60, best_prec_60, best_rec_60 = t, p, r

results_df = pd.DataFrame(results_table)

# Best F2 threshold (recall-weighted)
results_df['f2'] = (5 * results_df['precision'] * results_df['recall']) / (4 * results_df['precision'] + results_df['recall'] + 1e-10)
results_df['f1'] = (2 * results_df['precision'] * results_df['recall']) / (results_df['precision'] + results_df['recall'] + 1e-10)
best_f2_row = results_df.loc[results_df['f2'].idxmax()]

print('=' * 70)
if best_rec_60 > 0:
    print(f'Threshold precision >= {target_precision:.0%}: t={best_t_60:.4f}, '
          f'Prec={best_prec_60:.4f}, Rec={best_rec_60:.4f}')
    if best_rec_60 < 0.10:
        print(f'>> Recall {best_rec_60:.1%} es inutilizable en produccion.')
        print(f'>> Forzando threshold F2 para priorizar recall.')
else:
    print(f'NO SE ALCANZA precision >= {target_precision:.0%} en validacion.')

print(f'Threshold F2 (maximiza recall con precision razonable):')
print(f"  t={best_f2_row['threshold']:.4f}, Prec={best_f2_row['precision']:.4f}, "
      f"Rec={best_f2_row['recall']:.4f}, F2={best_f2_row['f2']:.4f}")

print(f'\nThreshold F1 (balanceado):')
best_f1_row = results_df.loc[results_df['f1'].idxmax()]
print(f"  t={best_f1_row['threshold']:.4f}, Prec={best_f1_row['precision']:.4f}, Rec={best_f1_row['recall']:.4f}, F1={best_f1_row['f1']:.4f}")

# Use F2 threshold as final (recall-first objective)
best_t = best_f2_row['threshold']
best_prec = best_f2_row['precision']
best_rec = best_f2_row['recall']
print(f'\nThreshold FINAL (F2): t={best_t:.4f}, Prec={best_prec:.4f}, Rec={best_rec:.4f}')
print(f'  Precision ~{best_prec:.0%}: de cada 100 alertas, ~{int(best_prec*100)} son fraude real.')
print(f'  Recall ~{best_rec:.0%}: detecta ~{int(best_rec*100)} de cada 100 fraudes.')
print('=' * 70)


### Curva Precision-Recall con threshold

In [ ]:

prec_curve, rec_curve, thr_curve = precision_recall_curve(y_val, yprob_val)
plt.figure(figsize=(10, 6))
plt.plot(rec_curve, prec_curve, 'b-', linewidth=2, label=f'{best_name} (PR-AUC={average_precision_score(y_val, yprob_val):.4f})')
plt.axhline(y=target_precision, color='r', linestyle='--', alpha=0.5, label=f'Precisión target={target_precision}')
plt.axvline(x=best_rec, color='g', linestyle='--', alpha=0.5, label=f'Recall en threshold={best_rec:.3f}')
plt.scatter([best_rec], [best_prec], color='darkgreen', s=100, zorder=5,
            label=f'Threshold={best_t:.3f} (Prec={best_prec:.3f}, Rec={best_rec:.3f})')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title(f'Curva PR — {best_name}', fontsize=14)
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Threshold table (sampled)
results_df = pd.DataFrame(results_table)
n_rows = len(results_df)
step = max(1, n_rows // 20)
print('Thresholds representativos:')
print(results_df[::step].round(4).to_string(index=False))


## 9. Evaluación Final en Test

Entrenamos modelo final con train+validation (opcional) y evaluamos en test con el threshold seleccionado.

In [ ]:

# Refit best model on train (or train+val) and predict test
model_final = best_model  # refit happens below

# Option A: Train only (no retrain needed, already fitted on train)
# Already fitted on X_train_s
yprob_test = model_final.predict_proba(X_test_s)[:, 1]

# Apply threshold
yp_test = (yprob_test >= best_t).astype(int)

print('=' * 60)
print('EVALUACIÓN FINAL EN TEST')
print('=' * 60)
print(f'Modelo: {best_name}')
print(f'Threshold: {best_t:.4f}')
print()
print(classification_report(y_test, yp_test, digits=4))
print()

cm = confusion_matrix(y_test, yp_test)
print('Matriz de confusión:')
print(f'              TN={cm[0,0]:,}   FP={cm[0,1]:,}')
print(f'              FN={cm[1,0]:,}   TP={cm[1,1]:,}')
print()
print(f'Fraudes detectados (TP):  {cm[1,1]:,} / {cm[1,0]+cm[1,1]:,} ({cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%)')
print(f'Falsos positivos (FP):    {cm[0,1]:,}')
print(f'Alertas totales:          {cm[0,1]+cm[1,1]:,}')
print(f'Precisión en test:        {cm[1,1]/(cm[0,1]+cm[1,1])*100:.1f}%')
print(f'Recall en test:           {cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%')


In [ ]:
# Metrics table
print('\n' + '=' * 60)
print('METRICAS COMPLETAS')
print('=' * 60)
best_idx = 0 if best_name == 'LightGBM' else 1
print(f'  PR-AUC (test):          {average_precision_score(y_test, yprob_test):.4f}')
print(f'  AUC-ROC (test):         {roc_auc_score(y_test, yprob_test):.4f}')
print(f'  F1-score (fraude):      {f1_score(y_test, yp_test):.4f}')
print(f'  Precision@Recall=0.9:   {results_val[best_idx]["Prec@Rec90"]:.4f}')
print(f'  Alertas/100k:           {(yp_test.sum() / len(yp_test) * 100000):.1f}')
print()
print(f'Threshold recomendado:    {best_t:.4f}')


## 10. Interpretabilidad

In [ ]:

if best_name == 'LightGBM':
    imp = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.booster_.feature_importance(importance_type='gain')
    })
else:
    imp = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    })

imp = imp.sort_values('importance', ascending=False).head(20)
imp['importance_pct'] = imp['importance'] / imp['importance'].sum() * 100

plt.figure(figsize=(10, 8))
sns.barplot(data=imp, y='feature', x='importance_pct', palette='viridis')
plt.xlabel('Importancia relativa (%)')
plt.ylabel('Feature')
plt.title(f'Top 20 Features — {best_name}')
plt.tight_layout()
plt.show()


## 11. Conclusiones y Recomendaciones

### Resumen
El reporte final con métricas en test se genera en la celda de código anterior.

### Interpretación operativa
- El modelo detecta la mayoría de los fraudes (recall reportado arriba).
- Los falsos positivos deben revisarse según el costo operativo de cada alerta.
- La precisión determina cuántas alertas son fraude real vs falso positivo.

### Mejoras propuestas para subir recall sin destruir precisión
1. **Más features derivadas:** features de red (conexiones entre cuentas), features temporales avanzadas (diferencias respecto a media móvil de 7 días), features de sesión (transacciones en ráfaga).
2. **Ensamblado calibrado:** promediar LightGBM + XGBoost con pesos óptimos puede dar mejor calibración que cada modelo solo.
3. **Threshold dinámico:** en producción, ajustar threshold según hora del día o tipo de transacción (más permisivo en horas de alta actividad).
4. **Selección de features con permutación:** eliminar features ruidosas para mejorar la relación señal/ruido.
5. **Datos reales vs sintéticos:** este dataset es generado con Faker; los patrones reales de fraude son más estructurados (redes, estacionalidad, tendencias). Con datos reales, el recall objetivo debería ser >95%.
6. **Revisión de falsos negativos:** analizar los fraudes no detectados (FN) para identificar patrones que el modelo no captura y crear features específicas.
7. **Monitoreo en producción:** la distribución de probabilidades debe monitorearse continuamente (PSI, drift) para reentrenar antes de que el threshold quede obsoleto.
